In [1]:
from models_v2 import *
from src.plots import *

In [2]:
import torch
import torch.nn as nn
import numpy as np
from tqdm import tqdm
from sklearn.metrics import confusion_matrix

from torch.utils.data import DataLoader
from datasets import ADE20KSegmentation
from torchinfo import summary

In [3]:
# setup
image_size = 224
num_classes = 150

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
data_root = "./images/ADE20K_2021_17_01/images/ADE"
save_model_path = "./first_try.pth"

In [5]:
print("Loading model: segmentation_deit_small_patch16_LS_reg")
# NOTE: we always used 224x224 images for deit so this has been set as default
image_size = 224
num_registers = 4

model = segmentation_deit_small_patch16_LS_reg(
    pretrained=True,
    # img_size=image_size, # default is 224
    pretrained_21k=True,
    num_registers=num_registers,
)

Loading model: segmentation_deit_small_patch16_LS_reg
>>>>>>>>>>>>>>>>>>>> PRETRAINED MODEL WILL BE USED
>>>>>>>>>>>>>>>>>>>> NUMBER OF REGISTERS: 4
Adding 4 register token(s) for fine-tuning.


In [6]:
model

DeitSegModel(
  (vit): vit_models(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))
      (norm): Identity()
    )
    (blocks): ModuleList(
      (0-11): 12 x Layer_scale_init_Block(
        (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=384, out_features=1152, bias=True)
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=384, out_features=384, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (drop_path): Identity()
        (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (mlp): Mlp(
          (fc1): Linear(in_features=384, out_features=1536, bias=True)
          (act): GELU(approximate='none')
          (fc2): Linear(in_features=1536, out_features=384, bias=True)
          (drop): Dropout(p=0.0, inplace=False)
        )
      )
    )
    (norm): LayerNorm((384,), e

In [7]:
freeze_layers = 12

if freeze_layers > 0:
    for param in model.vit.patch_embed.parameters():
        param.requires_grad = False
    print(">" * 20, "Froze the patch embeddings.")
    num_blocks = len(model.vit.blocks)
    freeze_until = min(freeze_layers, num_blocks)
    for block_idx in range(freeze_until):
        for param in model.vit.blocks[block_idx].parameters():
            param.requires_grad = False
    print(">" * 20, f"Froze {freeze_until} out of {num_blocks} layers of the model.")

summary(model)

>>>>>>>>>>>>>>>>>>>> Froze the patch embeddings.
>>>>>>>>>>>>>>>>>>>> Froze 12 out of 12 layers of the model.


Layer (type:depth-idx)                        Param #
DeitSegModel                                  --
├─vit_models: 1-1                             77,184
│    └─PatchEmbed: 2-1                        --
│    │    └─Conv2d: 3-1                       (295,296)
│    │    └─Identity: 3-2                     --
│    └─ModuleList: 2-2                        --
│    │    └─Layer_scale_init_Block: 3-3       (1,775,232)
│    │    └─Layer_scale_init_Block: 3-4       (1,775,232)
│    │    └─Layer_scale_init_Block: 3-5       (1,775,232)
│    │    └─Layer_scale_init_Block: 3-6       (1,775,232)
│    │    └─Layer_scale_init_Block: 3-7       (1,775,232)
│    │    └─Layer_scale_init_Block: 3-8       (1,775,232)
│    │    └─Layer_scale_init_Block: 3-9       (1,775,232)
│    │    └─Layer_scale_init_Block: 3-10      (1,775,232)
│    │    └─Layer_scale_init_Block: 3-11      (1,775,232)
│    │    └─Layer_scale_init_Block: 3-12      (1,775,232)
│    │    └─Layer_scale_init_Block: 3-13      (1,775,232)
│  

In [8]:
train_dataset = ADE20KSegmentation(data_root, is_train=True)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=0)

optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
criterion = nn.CrossEntropyLoss(ignore_index=255)

In [9]:
print("Fine-tuning...")
model.train()
for epoch in range(1):
    total_loss = 0
    for images, masks in tqdm(train_loader, desc=f"Epoch {epoch+1}/5"):
        images, masks = images.to(device), masks.to(device)
        preds = model(images)
        loss = criterion(preds, masks)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1} - Loss: {total_loss / len(train_loader):.4f}")

    torch.save(model.state_dict(), save_model_path)
    print(f"[EPOCH {epoch+1}] Model saved to {save_model_path}")

Fine-tuning...


Epoch 1/5:   0%|          | 0/136 [00:10<?, ?it/s]


KeyboardInterrupt: 

In [4]:
def compute_miou(preds, labels, num_classes, ignore_index=255):
    mask = labels != ignore_index
    preds = preds[mask]
    labels = labels[mask]
    cm = confusion_matrix(labels.flatten(), preds.flatten(), labels=list(range(num_classes)))
    intersection = np.diag(cm)
    union = np.sum(cm, axis=1) + np.sum(cm, axis=0) - np.diag(cm)
    iou = intersection / np.maximum(union, 1)
    return np.mean(iou), iou

def intersection_and_union(pred, label, num_classes):
    pred = np.asarray(pred).copy()
    label = np.asarray(label).copy()
    pred[label == 0] = 0 
    label[label == 0] = 0

    pred = pred * (label > 0)
    inter = pred * (pred == label)

    inter_area = np.histogram(inter, bins=num_classes, range=(1, num_classes))[0]
    pred_area = np.histogram(pred, bins=num_classes, range=(1, num_classes))[0]
    label_area = np.histogram(label, bins=num_classes, range=(1, num_classes))[0]
    union_area = pred_area + label_area - inter_area
    return inter_area, union_area

In [ ]:
# load model from save_model_path

model = segmentation_deit_small_patch16_LS_reg(
    pretrained=False,
    # img_size=image_size, # default is 224
    num_registers=num_registers,
)

model.load_state_dict(torch.load(save_model_path, map_location=device))
model.eval()
print("Model loaded from", save_model_path)

>>>>>>>>>>>>>>>>>>>> NUMBER OF REGISTERS: 4
Adding 4 register token(s) for fine-tuning.
Model loaded from ./first_try.pth


In [ ]:
# ---- Evaluation on full validation set ----
print("Evaluating on validation set...")
val_dataset = ADE20KSegmentation(data_root, is_train=False)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=0)

all_preds = []
all_labels = []

with torch.no_grad():
    for images, masks in tqdm(val_loader, desc="Evaluating"):
        images = images.to(device)
        logits = model(images)
        preds = logits.argmax(1).cpu()
        all_preds.append(preds)
        all_labels.append(masks)

all_preds = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()

miou, iou_per_class = compute_miou(all_preds, all_labels, num_classes=num_classes)
print(f"Validation mIoU: {miou:.4f}")

# ---- Evaluation on custom test image ----
# print("Evaluating custom test image...")
# image = Image.open(custom_image_path).convert("RGB")
# mask = Image.open(custom_mask_path).convert("L")

# transform = transforms.Compose([
#     transforms.Resize((image_size, image_size)),
#     transforms.ToTensor(),
# ])
# image_tensor = transform(image).unsqueeze(0).to(device)
# mask_np = np.array(mask.resize((image_size, image_size), resample=Image.NEAREST)).astype(np.int64)
# mask_np[(mask_np != 255) & (mask_np > 149)] = 149

# with torch.no_grad():
#     pred = model(image_tensor).argmax(1).squeeze(0).cpu().numpy()

# colormap = np.array(colormaps.ADE20K_COLORMAP, dtype=np.uint8)
# colored_pred = colormap[pred + 1]
# Image.fromarray(colored_pred).save(output_seg_path)

# miou_test, _ = compute_miou(pred, mask_np, num_classes=num_classes)
# print(f"Test image mIoU: {miou_test:.4f}")
# print(f"Segmentation saved at {output_seg_path}")

Evaluating on validation set...


/var/folders/cd/sr6rtt2j1rqdn0jdyblt3ztm0000gn/T/ipykernel_78228/3658614502.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(save_model_p

Validation mIoU: 0.0235


Fine tune + head: mIoU = 0.0235

Only head: 